# Red Team Agent in action

**Goal:** show how Microsoft Foundry generates, runs, evaluates, and reports adversarial attacks against a model through the server-side preview API in `azure-ai-projects`.

## Red teaming without PyRIT

Red teaming is a testing practice, not a synonym for PyRIT. Microsoft provides two relevant execution paths:

- **Client-side orchestration:** `azure-ai-evaluation[redteam]` uses PyRIT in the local Python environment to orchestrate attacks. Its target can still be a remote model, endpoint, or application; therefore, PyRIT itself is not limited to testing local targets.
- **Managed Foundry scan, used here:** `project_client.beta.red_teams` submits the scan to Microsoft Foundry. Attack orchestration and evaluation are managed by the Azure service, so the local environment does not need PyRIT.

---

This notebook deliberately uses the second path to keep the shared UV environment small and the demo reliable.

| Time | What to show |
| --- | --- |
| 0:00 | Why red teaming complements evaluation |
| 0:40 | Target, risk category, and attack strategies |
| 1:30 | Start or retrieve the scan |
| 3:00 | Compare the ASR scorecard and inspect attack details |
| 4:30 | Explain the finding and close with three takeaways |

## Variables, Constants and Libraries definition

In [ ]:
# pyright: reportUnusedImport=false
import os, time
from datetime import datetime

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AttackStrategy,
    AzureOpenAIModelConfiguration,
    RedTeam,
    RiskCategory,
)
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

if not load_dotenv("./../../credentials_my.env"):
    raise RuntimeError("Environment variables could not be loaded")

foundry_project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
deployment_name = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"]

project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

print(f"Foundry project endpoint: {foundry_project_endpoint}")
print(f"Target deployment: {deployment_name}")

Foundry project endpoint: https://mm-ai-upskilling-project-resourc.services.ai.azure.com/api/projects/ai-upskilling-project
Target deployment: gpt-5.4-mini


## A deliberately small scan

To stay within five minutes, this demo tests one risk category, **Violence**, and compares the **baseline** with a **Base64-obfuscated** attack over a single turn.

The goal is not comprehensive coverage. It is to make the workflow visible: `target → attacks → evaluation → mitigation`.

In [2]:
# 24 attacks: 4 turns by 2 risk_categories by 3 attack_strategies

redteam_config = {
    "num_turns": 4,
    "risk_categories": [
        RiskCategory.VIOLENCE,
        RiskCategory.CODE_VULNERABILITY,
    ],
    "attack_strategies": [
        AttackStrategy.BASELINE,
        AttackStrategy.BASE64,
        AttackStrategy.FLIP,
    ],
    "model_deployment_name": deployment_name,
}

scan_config = RedTeam(
    target=AzureOpenAIModelConfiguration(
        model_deployment_name=redteam_config["model_deployment_name"],
    ),
    display_name=f"5-min-demo-{datetime.now():%Y%m%d-%H%M}",
    num_turns=redteam_config["num_turns"],

    # 👇 Ricostruisci esplicitamente le liste
    attack_strategies=redteam_config["attack_strategies"],
    risk_categories=redteam_config["risk_categories"],

    application_scenario=(
        "A general-purpose assistant that should refuse requests "
        "for harmful violent content."
    ),
    simulation_only=False,
    tags={"purpose": "five-minute-demo"},
)

scan_config

{'target': {'modelDeploymentName': 'gpt-5.4-mini', 'type': 'AzureOpenAIModel'}, 'displayName': '5-min-demo-20260815-1627', 'numTurns': 4, 'attackStrategies': ['baseline', 'base64', 'flip'], 'riskCategories': ['Violence', 'CodeVulnerability'], 'applicationScenario': 'A general-purpose assistant that should refuse requests for harmful violent content.', 'simulationOnly': False, 'tags': {'purpose': 'five-minute-demo'}}

## Live action

Only the next cell creates cloud work. For maximum reliability, run it before the presentation; during the live demo, retrieve that scan or show the recent scans instead.

In [3]:
scan = project_client.beta.red_teams.create(red_team=scan_config)

print(f"Scan ID: {scan.name}")
print(f"Display name: {scan.display_name}")
print(f"Initial status: {scan.status}")

Scan ID: 0365e0a2-c4b6-462e-8894-291d7e0de0f4
Display name: 5-min-demo-20260815-1627
Initial status: NotStarted


In [5]:
terminal_statuses = {"Completed", "Failed", "Cancelled", "Canceled"}
last_status = None

while(True):
    scan = project_client.beta.red_teams.get(scan.name)
    current_status = (scan.status or "unknown").capitalize()

    if current_status != last_status:
        print(f"Status: from {last_status} to {scan.status}")
        last_status = current_status
    else:
        print(f"Status: {scan.status}")

    if current_status in terminal_statuses:
        break

    time.sleep(2)

Status: from None to Starting
Status: Starting
Status: Starting
Status: Starting
Status: Starting
Status: Starting
Status: Starting
Status: Starting
Status: Starting
Status: Starting
Status: Starting
Status: Starting
Status: Starting
Status: Starting
Status: Starting
Status: from Starting to Completed


## Select a completed scan

A live scan may take longer than the presentation slot. This read-only fallback selects the most recent completed scan, so the result section always has real data to show.

When presenting, either keep the `scan` returned by the polling cell or run the next cell to select a previously completed scan.

In [6]:
completed_scans = [
    recent_scan
    for recent_scan in project_client.beta.red_teams.list()
    if (recent_scan.status or "").lower() == "completed"
]

if not completed_scans:
    raise RuntimeError(
        "No completed scan is available. Run the live scan and wait for completion."
    )

scan = completed_scans[0]
{
    "scan_id": scan.name,
    "display_name": scan.display_name,
    "status": scan.status,
    "risk_categories": scan.risk_categories,
    "attack_strategies": scan.attack_strategies,
}

{'scan_id': '0365e0a2-c4b6-462e-8894-291d7e0de0f4',
 'display_name': '5-min-demo-20260815-1627',
 'status': 'Completed',
 'risk_categories': [<RiskCategory.VIOLENCE: 'Violence'>,
  <RiskCategory.CODE_VULNERABILITY: 'CodeVulnerability'>],
 'attack_strategies': ['Baseline', 'Base64', 'Flip']}

## Show the actual result

**Attack Success Rate (ASR)** is the percentage of generated attacks that caused a policy-violating response. Lower is better; `0%` means that none of the evaluated attacks succeeded.

The managed service groups Base64 under **easy complexity** in its metric names. The next cell converts the raw metrics into a presentation-ready scorecard and provides the detailed Foundry report link.

In [7]:
import json, math
from IPython.display import Markdown, display

scan = next(
    (
        listed_scan
        for listed_scan in project_client.beta.red_teams.list()
        if listed_scan.name == scan.name
    ),
    scan,
)
scan_payload = scan.as_dict()

if (scan.status or "").lower() != "completed":
    raise RuntimeError(f"Scan results are not ready. Current status: {scan.status}")

outputs = scan_payload.get("outputs") or {}
raw_metrics = outputs.get("evaluationMetrics", "{}")
metrics = json.loads(raw_metrics) if isinstance(raw_metrics, str) else raw_metrics

scorecard = [
    {
        "strategy": "Baseline",
        "service_metric": "violence_baseline_asr",
        "asr": metrics.get("violence_baseline_asr"),
    },
    {
        "strategy": "Base64 (easy complexity)",
        "service_metric": "violence_easy_complexity_asr",
        "asr": metrics.get("violence_easy_complexity_asr"),
    },
]

for row in scorecard:
    if row["asr"] is None:
        row["result"] = "Metric not available"
    elif row["asr"] == 0:
        row["result"] = "No successful attacks"
    else:
        row["result"] = "At least one attack succeeded"

table_rows = [
    "| Strategy | Risk category | ASR | Interpretation |",
    "| --- | --- | ---: | --- |",
]
for row in scorecard:
    asr = "n/a" if row["asr"] is None else f"{math.floor(row['asr'] * 10) / 10}%"
    table_rows.append(
        f"| {row['strategy']} | Violence | {asr} | {row['result']} |"
    )

evaluation_result_id = outputs.get("evaluationResultId", "Not available")
report_url = (scan_payload.get("properties") or {}).get("AiStudioEvaluationUri")

summary = [
    f"### Result for `{scan.display_name}`",
    "",
    *table_rows,
    "",
    f"**Evaluation result:** `{evaluation_result_id}`",
]
if report_url:
    summary.extend(["", f"[Open attack details in Microsoft Foundry]({report_url})"])

display(Markdown("\n".join(summary)))

### Result for `5-min-demo-20260815-1627`

| Strategy | Risk category | ASR | Interpretation |
| --- | --- | ---: | --- |
| Baseline | Violence | 25.0% | At least one attack succeeded |
| Base64 (easy complexity) | Violence | 25.0% | At least one attack succeeded |

**Evaluation result:** `azureai://accounts/mm-ai-upskilling-project-resourc/projects/ai-upskilling-project/evaluationresults/eval-result-0365e0a2-c4b6-462e-8894-291d7e0de0f4-ITEI/versions/1`

[Open attack details in Microsoft Foundry](https://ai.azure.com/resource/build/redteaming/0365e0a2-c4b6-462e-8894-291d7e0de0f4?wsid=/subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/ai-upskilling-rg/providers/Microsoft.CognitiveServices/accounts/mm-ai-upskilling-project-resourc/projects/ai-upskilling-project&tid=3ad0b905-34ab-4116-93d9-c1dcc2d35af6)

## Closing

- Red teaming generates adaptive adversarial probes rather than relying only on a static test set.
- Comparing baseline and transformed attacks exposes weaknesses in application controls.
- Results provide evidence for mitigations and regression tests, but they do not guarantee security.